### Introduction to Bayes' Theorem: The Logic of Science
Is a mathematical formula used to update the probability of a hypothesis as more evidence or information becomes available.
Instead of guessing in a vacuum, it allows us to combine our **prior knowledge** with **new data** to make highly accurate predictions.


#### The Core Formula

The theorem is mathematically expressed as:

$$P(A|B) = \frac{P(B|A) \times P(A)}{P(B)}$$

Where each term represents a critical piece of data science intuition:

*   **$P(A|B)$ (Posterior Probability):** The probability of event $A$ occurring *given* that event $B$ has already happened. This is what we want to find out (e.g., *What is the chance a server is failing given that an alert triggered?*).
*   **$P(B|A)$ (Likelihood):** The probability of event $B$ occurring *given* that event $A$ is true. This represents historical behavior (e.g., *If a server actually fails, how often does our system correctly catch it?*).
*   **$P(A)$ (Prior Probability):** The baseline probability of event $A$ occurring before looking at any new evidence. This is our foundational base rate (e.g., *The general failure rate of hardware in our data center*).
*   **$P(B)$ (Marginal Probability):** The total probability of event $B$ occurring across the entire population, whether event $A$ happens or not (e.g., *The total number of alerts generated by all systems combined*).


### Case Study: The False Positive Paradox in Drug Testing
Even a **"highly accurate"** test can produce more false positives than true positives when looking for a rare event in a population.
#### Given Parameters:
* **Base Rate of Usage:** Only `0.3%` of the overall population actually uses this drug.
  * $P(A) = 0.003$ (Probability of being a user)
  * $P(\text{Not } A) = 0.997$ (Probability of being a non-user)
* **Test Accuracy for Users (Sensitivity / Recall):** The test accurately identifies users `99%` of the time.
  * $P(B|A) = 0.99$ (Probability of testing positive *given* you are a user)
* **Test Accuracy for Non-Users (Specificity):** The test accurately returns a negative result `99%` of the time for non-users.
  * This implies a `1%` false positive rate: $P(B|\text{Not } A) = 0.01$ (Probability of testing positive *given* you are a non-user)



#### Step 1: Calculate the Total Probability of Testing Positive $P(B)$
We arrive at $P(B) = 1.3\%$ by summing the probability of testing positive if you do use the drug, plus the probability of testing positive if you do not:

$$P(B) = (0.99 \times 0.003) + (0.01 \times 0.997) = 0.00297 + 0.00997 = 0.01294 \approx 1.3\%$$

---

#### Step 2: Apply Bayes' Theorem
To find the probability that someone is an actual user given that they tested positive ($P(A|B)$), we use the math represented below:

$$P(A|B) = \frac{P(A)P(B|A)}{P(B)} = \frac{0.003 \times 0.99}{0.013} = 22.8\%$$

#### Conclusion:
* So the odds of someone being an actual user of the drug given that they tested positive is **only 22.8%**!
* Even though $P(B|A)$ is exceptionally high (`99%`), it does **not** mean $P(A|B)$ is high.

In [ ]:
import os
import io
import numpy
import pandas as pd
from pandas import DataFrame


In [ ]:
import os
import io
import zipfile
import numpy
import pandas as pd
from pandas import DataFrame

# ============================================================
# [FIXED / IMPROVED IMPLEMENTATION]:
# 1. FIXED PATH ISSUE: The pulled code assumed 'emails/spam' is always relative to cwd.
#    If the notebook is run from the project root, it would crash because the directory
#    lives under 'Day6/emails/'. This helper dynamically resolves the correct path.
# 2. FIXED ZIP EXTRACTION: If 'emails.zip' has not been extracted yet, we automatically
#    extract it so the notebook never fails with FileNotFoundError or empty vocabulary.
# 3. FIXED VARIABLE SHADOWING: 'path' argument in readFiles was shadowed by 'path = os.path.join(...)'.
# 4. FIXED RESOURCE LEAK: Switched to 'with io.open(...) as f:' for safe descriptor closing.
# ============================================================

def resolve_emails_directory():
    '''Locates or extracts the emails directory regardless of current working directory.'''
    candidates = ['Day6/emails', 'emails', '../emails']
    for c in candidates:
        if os.path.exists(c) and os.path.exists(os.path.join(c, 'spam')):
            return c
    
    # Check for emails.zip to auto-extract
    zip_candidates = ['Day6/emails.zip', 'emails.zip', '../emails.zip']
    for zpath in zip_candidates:
        if os.path.exists(zpath):
            target_dir = os.path.dirname(zpath)
            print(f"Auto-extracting '{zpath}' into '{target_dir}'...")
            with zipfile.ZipFile(zpath, 'r') as zf:
                zf.extractall(target_dir if target_dir else '.')
            return os.path.join(target_dir if target_dir else '.', 'emails')
    
    raise FileNotFoundError("Could not find 'emails' directory or 'emails.zip'.")

emails_base = resolve_emails_directory()

def readFiles(directory_path):
    '''Walks directory and safely extracts email message bodies.'''
    for root, dirnames, filenames in os.walk(directory_path):
        for filename in filenames:
            file_full_path = os.path.join(root, filename)
            inBody = False
            lines = []
            with io.open(file_full_path, 'r', encoding='latin1') as f:
                for line in f:
                    if inBody:
                        lines.append(line)
                    elif line == '\n':
                        inBody = True
            message = '\n'.join(lines)
            yield file_full_path, message


def dataFrameFromDirectory(directory_path, classification):
    '''Constructs DataFrame from all emails in directory.'''
    rows = []
    index = []
    for filename, message in readFiles(directory_path):
        rows.append({'message': message, 'class': classification})
        index.append(filename)
    return DataFrame(rows, index=index)

spam_dir = os.path.join(emails_base, 'spam')
ham_dir = os.path.join(emails_base, 'ham')

data = DataFrame({'message': [], 'class': []})
data = pd.concat([data, dataFrameFromDirectory(spam_dir, "spam")])
data = pd.concat([data, dataFrameFromDirectory(ham_dir, "ham")])
print(f"Loaded {len(data)} total emails ({len(data[data['class']=='spam'])} spam, {len(data[data['class']=='ham'])} ham)")


In [ ]:
print(data.head())
print(data.tail())

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()
counts = vectorizer.fit_transform(data['message'].values)
# print(counts)

In [6]:
print("_" * 60)
print(counts.toarray())

____________________________________________________________
[[1 0 1 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


In [7]:
# Convert sparse matrix into a labeled DataFrame
counts_df = pd.DataFrame(
    counts.toarray(), 
    columns=vectorizer.get_feature_names_out()
)

# Display the first few rows
counts_df.head()

,00,000,0000,000000,000000000,000000000000000000000,00000254,000033,000058,000066,...,þàì,þàìå,þàï,þàïà,þàïàº,þàïáö¼ò,þîñ,þõµ,þüg,ÿÿÿÿÿÿÿÿÿÿÿÿÿÿÿÿÿÿÿÿÿÿÿÿÿÿÿÿÿÿÿÿÿÿÿÿÿÿÿÿÿó
0,1,0,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [8]:
# Print the first 5 emails and first 10 word columns only
print(counts[:5, :10].toarray())

[[1 0 1 1 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]]


In [9]:
# 1. Look at columns in the middle of the DataFrame (where alphabetical words live)
print(counts_df.iloc[:5, 10000:10010])

   ah634vnghvsegfqk  ah8uds7v2msdnnhdbz0k  ah9inw0tusq2p4t4tpksek4fjw  aha  \
0                 0                     0                           0    0   
1                 0                     0                           0    0   
2                 0                     0                           0    0   
3                 0                     0                           0    0   
4                 0                     0                           0    0   

   aha7dnjnywlhtghliif0z  ahahealth  ahamembership  ahclxz08dnnutrpypq  \
0                      0          0              0                   0   
1                      0          0              0                   0   
2                      0          0              0                   0   
3                      0          0              0                   0   
4                      0          0              0                   0   

   ahdqftbcuyk1jvkkzvz3dp2fbld0yy7uvt3v  ahdtdxh2cnn5cwn3amh2dhr4dzwvdgl0bgu  
0      

In [10]:
# 2. Or check count for specific words if they exist in vocabulary
words_to_check = ['free', 'subject', 'meeting', 'money']
existing_words = [w for w in words_to_check if w in counts_df.columns]

In [11]:
print(counts_df[existing_words].head())

   free  subject  meeting  money
0     3        0        0      1
1     1        0        0      0
2     1        0        0      0
3    14        0        0      0
4     1        0        0      0


In [12]:
from sklearn.naive_bayes import MultinomialNB

In [13]:
classifier = MultinomialNB()
targets = data['class'].values
classifier.fit(counts, targets)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](2,)","[2500., 500.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](2,)","[-0.18,-1.79]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[<U4](2,)","['ham','spam']"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](2, 62964)","[[855.,319., 52.,..., 0., 0., 0.], [300.,632., 21.,..., 1., 1., 1.]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](2, 62964)","[[ -6.57, -7.55, -9.35,...,-13.32,-13.32,-13.32], [ -7.13, -6.39, -9.75,...,-12.15,-12.15,-12.15]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,62964


In [19]:
# 1. New test emails
# examples = [
#     "Are we still having our team meeting today?",
#     "Free claim your cash prize now", 
# ]
# examples = ['Free Viagra now!!!', "Hi Bob, how about a game of golf tomorrow?"]

examples = ["Free Free Free Loan To london","Free loan"]

# 2. Transform the new examples into the same feature space as the training data
examples_counts = vectorizer.transform(examples)

# 3. Predict the class for each new example
predictions = classifier.predict(examples_counts)
print(predictions)

['spam' 'spam']


In [20]:
# another example

# 1. Build Static Dataset of 30 Cricketers
cricketers_data = [
    # --- Batsmen ---
    {"name": "Sachin Tendulkar", "runs": 18426, "wickets": 154, "batting_avg": 44.83, "bowling_avg": 44.48, "catches": 140, "role": "Batsman"},
    {"name": "Virat Kohli",      "runs": 13848, "wickets": 5,   "batting_avg": 58.18, "bowling_avg": 166.20, "catches": 152, "role": "Batsman"},
    {"name": "Rohit Sharma",     "runs": 10866, "wickets": 8,   "batting_avg": 49.16, "bowling_avg": 64.37,  "catches": 93,  "role": "Batsman"},
    {"name": "Ricky Ponting",    "runs": 13704, "wickets": 3,   "batting_avg": 42.03, "bowling_avg": 34.66,  "catches": 160, "role": "Batsman"},
    {"name": "Kumar Sangakkara", "runs": 14234, "wickets": 0,   "batting_avg": 41.98, "bowling_avg": 0.0,    "catches": 402, "role": "Batsman"},
    {"name": "Brian Lara",       "runs": 10405, "wickets": 4,   "batting_avg": 40.48, "bowling_avg": 152.25, "catches": 120, "role": "Batsman"},
    {"name": "Joe Root",         "runs": 6522,  "wickets": 38,  "batting_avg": 34.87, "bowling_avg": 58.05,  "catches": 88,  "role": "Batsman"},
    {"name": "Steve Smith",      "runs": 5446,  "wickets": 28,  "batting_avg": 43.91, "bowling_avg": 34.67,  "catches": 82,  "role": "Batsman"},
    {"name": "Kane Williamson",  "runs": 6810,  "wickets": 7,   "batting_avg": 48.64, "bowling_avg": 35.42,  "catches": 80,  "role": "Batsman"},
    {"name": "David Warner",     "runs": 6932,  "wickets": 0,   "batting_avg": 45.30, "bowling_avg": 0.0,    "catches": 75,  "role": "Batsman"},

    # --- Bowlers ---
    {"name": "Muttiah Muralitharan", "runs": 674,  "wickets": 534, "batting_avg": 6.80,  "bowling_avg": 23.08, "catches": 130, "role": "Bowler"},
    {"name": "Wasim Akram",          "runs": 3717, "wickets": 502, "batting_avg": 16.52, "bowling_avg": 23.52, "catches": 88,  "role": "Bowler"},
    {"name": "Glenn McGrath",        "runs": 115,  "wickets": 381, "batting_avg": 3.83,  "bowling_avg": 22.02, "catches": 36,  "role": "Bowler"},
    {"name": "Anil Kumble",          "runs": 938,  "wickets": 337, "batting_avg": 10.53, "bowling_avg": 30.89, "catches": 85,  "role": "Bowler"},
    {"name": "Shane Warne",          "runs": 1018, "wickets": 293, "batting_avg": 13.05, "bowling_avg": 25.72, "catches": 80,  "role": "Bowler"},
    {"name": "Jasprit Bumrah",       "runs": 110,  "wickets": 149, "batting_avg": 6.87,  "bowling_avg": 23.55, "catches": 22,  "role": "Bowler"},
    {"name": "Mitchell Starc",       "runs": 551,  "wickets": 236, "batting_avg": 12.52, "bowling_avg": 23.12, "catches": 48,  "role": "Bowler"},
    {"name": "Trent Boult",          "runs": 190,  "wickets": 211, "batting_avg": 9.50,  "bowling_avg": 25.70, "catches": 38,  "role": "Bowler"},
    {"name": "Kagiso Rabada",        "runs": 350,  "wickets": 157, "batting_avg": 11.66, "bowling_avg": 27.20, "catches": 28,  "role": "Bowler"},
    {"name": "Lasith Malinga",       "runs": 567,  "wickets": 338, "batting_avg": 6.44,  "bowling_avg": 28.87, "catches": 31,  "role": "Bowler"},

    # --- Allrounders ---
    {"name": "Jacques Kallis",   "runs": 11579, "wickets": 273, "batting_avg": 44.36, "bowling_avg": 31.79, "catches": 131, "role": "Allrounder"},
    {"name": "Sanath Jayasuriya","runs": 13430, "wickets": 323, "batting_avg": 32.36, "bowling_avg": 36.75, "catches": 123, "role": "Allrounder"},
    {"name": "Shahid Afridi",    "runs": 8064,  "wickets": 395, "batting_avg": 23.57, "bowling_avg": 34.51, "catches": 127, "role": "Allrounder"},
    {"name": "Kapil Dev",        "runs": 3783,  "wickets": 253, "batting_avg": 23.79, "bowling_avg": 27.45, "catches": 71,  "role": "Allrounder"},
    {"name": "Ravindra Jadeja",  "runs": 2756,  "wickets": 220, "batting_avg": 32.42, "bowling_avg": 36.07, "catches": 73,  "role": "Allrounder"},
    {"name": "Hardik Pandya",    "runs": 1769,  "wickets": 84,  "batting_avg": 34.01, "bowling_avg": 35.61, "catches": 32,  "role": "Allrounder"},
    {"name": "Ben Stokes",       "runs": 3463,  "wickets": 74,  "batting_avg": 38.91, "bowling_avg": 42.39, "catches": 50,  "role": "Allrounder"},
    {"name": "Shakib Al Hasan",  "runs": 7570,  "wickets": 317, "batting_avg": 37.29, "bowling_avg": 29.52, "catches": 58,  "role": "Allrounder"},
    {"name": "Shane Watson",     "runs": 5757,  "wickets": 168, "batting_avg": 40.54, "bowling_avg": 31.79, "catches": 64,  "role": "Allrounder"},
    {"name": "Chris Cairns",     "runs": 4950,  "wickets": 201, "batting_avg": 29.46, "bowling_avg": 32.80, "catches": 66,  "role": "Allrounder"}
]

# 2. Convert into Pandas DataFrame
df = pd.DataFrame(cricketers_data)
feature_cols = ['runs', 'wickets', 'batting_avg', 'bowling_avg', 'catches']
X_train = df[feature_cols]
y_train = df['role']

# 4. Train Multinomial Naive Bayes Model
classifier = MultinomialNB()
classifier.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](3,)","[10.,10.,10.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](3,)","[-1.1,-1.1,-1.1]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[<U10](3,)","['Allrounder','Batsman','Bowler']"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](3, 5)","[[ 63121. , 2308. , 336.71, 338.68, 795. ], [107193. , 247. , 449.38, 590.1 , 1392. ], [ 8230. , 3138. , 97.72, 253.67, 586. ]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](3, 5)","[[-0.06,-3.37,-5.29,-5.28,-4.43], [-0.02,-6.09,-5.5 ,-5.23,-4.37], [-0.4 ,-1.37,-4.83,-3.88,-3.04]]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](5,)","['runs','wickets','batting_avg','bowling_avg','catches']"


In [21]:
# 5. Define New/Unseen Cricketer Data to Classify
new_cricketers = pd.DataFrame([
    # Player 1: High runs, high batting avg, almost zero wickets -> Should be Batsman
    {"name": "New Player A", "runs": 8500, "wickets": 2, "batting_avg": 51.2, "bowling_avg": 110.0, "catches": 65},
    
    # Player 2: Low runs, high wickets, low bowling avg -> Should be Bowler
    {"name": "New Player B", "runs": 450, "wickets": 280, "batting_avg": 8.5, "bowling_avg": 24.1, "catches": 30},
    
    # Player 3: Good runs AND high wickets -> Should be Allrounder
    {"name": "New Player C", "runs": 5200, "wickets": 210, "batting_avg": 33.5, "bowling_avg": 30.2, "catches": 55}
])

# 6. Predict Roles
predictions = classifier.predict(new_cricketers[feature_cols])
# new player df
new_players = pd.DataFrame(new_cricketers)
new_players['predicted_role'] = predictions
print(new_players)

           name  runs  wickets  batting_avg  bowling_avg  catches  \
0  New Player A  8500        2         51.2        110.0       65   
1  New Player B   450      280          8.5         24.1       30   
2  New Player C  5200      210         33.5         30.2       55   

  predicted_role  
0        Batsman  
1         Bowler  
2     Allrounder  
